# 03 — Lexical retrieval and fusion

**Concept:** BM25 (term overlap) and Reciprocal Rank Fusion of BM25 + dense.

## The prediction — written before running

> **Genuinely unsure.** On the source system (1,147 investor profiles), RRF fusion
> *hurt* recall. But BM25 is strong on domain-specific terminology, which is exactly
> what NFCorpus is made of, and RRF is widely reported to help on BEIR-type data.
> The original null may have been a fact about that corpus, not about RRF.

That uncertainty is the honest state of knowledge, and it's the interesting case:
whichever way this resolves, we learn whether the source system's null was about the
*method* or about the *corpus*.

**Mechanisms in play:**
- *For BM25 here:* bio-medical queries carry rare exact terms ("statin",
  "aflatoxin") that a 384-dim embedder blurs and an inverted index nails.
- *For RRF here:* fusion helps when the two retrievers make **different** mistakes;
  dense and lexical errors on scientific text are plausibly decorrelated.
- *Against RRF (the source-system mechanism):* rank fusion discards score magnitudes.
  When one retriever is clearly stronger, averaging it with a weaker one dilutes it.

There's also a **method sub-question** parked in BUILD.md: RRF can fuse *full*
rankings (the original formulation) or *truncated* candidate lists (what production
systems do — each retriever contributes only its top-`depth`). Rather than pick a
convention by fiat, `rrf()` takes a `depth` parameter and we measure the difference.

In [1]:
import numpy as np

from ragexp.data import load_nfcorpus
from ragexp.embed import Embedder
from ragexp.retrieve import dense, build_bm25, bm25, rrf
from ragexp.runs import evaluate_run, load_scores, metric_vector, save_scores, summarize
from ragexp.metrics import paired_bootstrap

corpus = load_nfcorpus()
qids = [q.query_id for q in corpus.queries]
dense_scores = load_scores("dense")  # the committed nb 02 baseline
print(corpus.summary())

NFCorpus: 3633 docs, 323 queries, 12334 judgments (38.2/query), grade distribution {1: 11758, 2: 576}


## Run BM25

Same signature as `dense()`: query in, full ranking out. Tokenization is
deliberately naive (`lower().split()`) — the lab measures methods, not preprocessing.

In [2]:
index, ids = build_bm25(corpus.docs)
bm25_rankings = {q.query_id: bm25(q.text, index, ids) for q in corpus.queries}
bm25_scores = evaluate_run(corpus, bm25_rankings, recall_ks=(10, 50), ndcg_ks=(10,))
save_scores("bm25", bm25_scores)
print({m: round(v, 4) for m, v in summarize(bm25_scores).items()})

{'recall@10': 0.1241, 'recall@50': 0.1792, 'ndcg@10': 0.2678}


## Fuse with RRF — at three truncation depths

Dense rankings are recomputed here (cache makes it cheap) because fusion needs the
*rankings*, not the saved scores. Fusion at `depth=None` (full 3,633-doc lists),
`depth=1000` (TREC-run convention), and `depth=100` (a tight production candidate
pool).

In [3]:
emb = Embedder()
doc_matrix = emb.encode([d.full for d in corpus.docs])
dense_rankings = {q.query_id: dense(q.text, corpus.doc_ids, doc_matrix, emb)
                  for q in corpus.queries}

rrf_scores = {}
for depth in (None, 1000, 100):
    rankings = {qid: rrf([dense_rankings[qid], bm25_rankings[qid]], depth=depth)
                for qid in qids}
    rrf_scores[depth] = evaluate_run(corpus, rankings, recall_ks=(10, 50), ndcg_ks=(10,))

save_scores("rrf", rrf_scores[1000])  # headline config: the production-shaped one

for depth, s in rrf_scores.items():
    print(f"depth={str(depth):>5}: {({m: round(v, 4) for m, v in summarize(s).items()})}")

depth= None: {'recall@10': 0.1447, 'recall@50': 0.2525, 'ndcg@10': 0.309}
depth= 1000: {'recall@10': 0.1477, 'recall@50': 0.2521, 'ndcg@10': 0.3093}
depth=  100: {'recall@10': 0.1469, 'recall@50': 0.2458, 'ndcg@10': 0.3106}


## The scoreboard — paired against the same 323 queries

Every comparison below is `paired_bootstrap` per notebook 01: per-query differences,
resampled, 95% CI, two-sided p. Positive delta = the second system is better.

In [4]:
def compare(name_a, scores_a, name_b, scores_b):
    print(f"{name_b} vs {name_a}")
    for metric in ("recall@10", "recall@50", "ndcg@10"):
        a = metric_vector(scores_a, metric, qids)
        b = metric_vector(scores_b, metric, qids)
        delta, (lo, hi), p = paired_bootstrap(a, b)
        sig = "  <-- significant" if (lo > 0 or hi < 0) else ""
        print(f"  {metric:>10}: delta {delta:+.4f}  CI [{lo:+.4f}, {hi:+.4f}]  p={p:.4f}{sig}")
    print()

compare("dense", dense_scores, "bm25", bm25_scores)
compare("dense", dense_scores, "rrf(depth=1000)", rrf_scores[1000])
compare("bm25", bm25_scores, "rrf(depth=1000)", rrf_scores[1000])

bm25 vs dense
   recall@10: delta -0.0309  CI [-0.0484, -0.0150]  p=0.0000  <-- significant
   recall@50: delta -0.0716  CI [-0.0953, -0.0500]  p=0.0000  <-- significant
     ndcg@10: delta -0.0489  CI [-0.0715, -0.0275]  p=0.0000  <-- significant

rrf(depth=1000) vs dense
   recall@10: delta -0.0073  CI [-0.0212, +0.0051]  p=0.2696
   recall@50: delta +0.0013  CI [-0.0112, +0.0132]  p=0.8286
     ndcg@10: delta -0.0074  CI [-0.0241, +0.0087]  p=0.3722

rrf(depth=1000) vs bm25
   recall@10: delta +0.0236  CI [+0.0139, +0.0348]  p=0.0000  <-- significant
   recall@50: delta +0.0729  CI [+0.0555, +0.0923]  p=0.0000  <-- significant
     ndcg@10: delta +0.0415  CI [+0.0297, +0.0535]  p=0.0000  <-- significant



## The depth sub-question

Does truncating the fused lists change anything at the ks we score? The mechanism
says it shouldn't much at `depth=1000` (we score at k ≤ 50, far above the truncation
point's influence) and might at `depth=100`.

In [5]:
for depth in (1000, 100):
    print(f"rrf(depth={depth}) vs rrf(depth=None)")
    for metric in ("recall@10", "recall@50", "ndcg@10"):
        a = metric_vector(rrf_scores[None], metric, qids)
        b = metric_vector(rrf_scores[depth], metric, qids)
        delta, (lo, hi), p = paired_bootstrap(a, b)
        sig = "  <-- significant" if (lo > 0 or hi < 0) else ""
        print(f"  {metric:>10}: delta {delta:+.4f}  CI [{lo:+.4f}, {hi:+.4f}]  p={p:.4f}{sig}")
    print()

rrf(depth=1000) vs rrf(depth=None)
   recall@10: delta +0.0030  CI [-0.0011, +0.0102]  p=0.4140
   recall@50: delta -0.0004  CI [-0.0020, +0.0009]  p=0.5904


     ndcg@10: delta +0.0003  CI [-0.0017, +0.0029]  p=0.8572

rrf(depth=100) vs rrf(depth=None)
   recall@10: delta +0.0022  CI [-0.0045, +0.0105]  p=0.5920
   recall@50: delta -0.0067  CI [-0.0151, -0.0004]  p=0.0302  <-- significant
     ndcg@10: delta +0.0016  CI [-0.0020, +0.0053]  p=0.3936



## Resolving the prediction

The prediction was **"genuinely unsure"**, with three mechanisms in play. Written
after seeing the numbers — which is fine, because the prediction and its mechanisms
were committed before any of this ran. Here's how each resolved:

1. **"BM25 strong on bio-medical terminology" — failed.** BM25 loses to dense on
   every metric (nDCG@10 −0.049, recall@50 −0.072, both p ≈ 0). Rare exact terms
   didn't save it: NFCorpus queries are natural-language questions
   ("*Do Cholesterol Statin Drugs Cause Breast Cancer?*"), not keyword queries, and
   naive tokenization gives BM25 no help with morphology. Lexical signal is real but
   strictly weaker than the embedder here.

2. **"RRF helps on BEIR-type data" — failed. The source system's null transferred.**
   RRF vs dense is a null on all three metrics (every CI straddles zero). Fusion
   comfortably beats the *weaker* parent (RRF vs BM25: +0.042 nDCG@10, p ≈ 0), but
   that's the trivial direction — nobody ships the weaker parent. The dilution
   mechanism is the reading: when one retriever is clearly stronger, rank-averaging
   it with a weaker one hands back roughly the stronger one. On the source system —
   where the gap was presumably larger — the same mechanism pushed the delta
   negative. Here it lands on zero.

3. **The depth sub-question (BUILD.md parking lot) — resolved empirically.**
   `depth=1000` is indistinguishable from fusing full rankings. `depth=100` costs a
   small but significant slice of recall@50 (−0.007, p = 0.03) — exactly the
   tail-clipping the mechanism predicts, since truncation at 100 removes
   contributions that matter near k = 50. Convention adopted: **`depth=1000`** —
   production-shaped, measurably lossless at the ks we score.

**Verdict for LESSONS.md:** hybrid retrieval is not a free lunch. RRF is a hedge
that costs you nothing when retrievers are comparable and buys you nothing when they
aren't — and "widely reported to help on BEIR" did not survive contact with this
corpus + this embedder.

Next: notebook 04, the spine — reranking, and the metric it cannot game.